# 1. Install Dependencies

In [ ]:
!pip install torch torchvision tqdm matplotlib numpy --quiet

# 2. Imports

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import math
import copy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

# 3. Config

In [ ]:
CONFIG = {
    'T': 500,
    'epochs': 5000,
    'lr': 2e-4,
    'batch_size': 256,
    'base_channels': 128,
    'num_classes': 10,
    'num_generated': 500,
    'num_workers': 2,
    'ckpt_epochs': [1000, 2000, 3000, 4000, 5000],
}

EXP_A_SIZE   = 1000
EXP_B_SIZES  = [50, 100, 250, 500, 1000, 2000, 5000]
EXP_C_SIZE   = 1000
EXP_D_SIZE   = 1000
EXP_D_C_VALS = [1, 10, 50, 100, 500, 1000]

# 4. Noise Schedule

In [ ]:
T = CONFIG['T']

beta      = torch.linspace(1e-4, 0.02, T, device=device)
alpha     = 1.0 - beta
alpha_hat = torch.cumprod(alpha, dim=0)   

def q_sample(x0, t):
    noise            = torch.randn_like(x0)
    sqrt_ahat        = alpha_hat[t].sqrt()[:, None, None, None]
    sqrt_one_m_ahat  = (1.0 - alpha_hat[t]).sqrt()[:, None, None, None]
    return sqrt_ahat * x0 + sqrt_one_m_ahat * noise, noise

# 5. Dataset Helpers

In [ ]:
_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
_cifar = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=_transform)

class LabelledSubset(Dataset):
    def __init__(self, indices: np.ndarray, labels: np.ndarray):
        assert len(indices) == len(labels)
        self.indices = indices
        self.labels  = labels.astype(np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        img, _ = _cifar[int(self.indices[i])]
        return img, int(self.labels[i])

def make_normal_subset(size: int, seed: int = 42) -> LabelledSubset:

    rng = np.random.RandomState(seed)
    idx = rng.choice(len(_cifar), size=size, replace=False)
    lbl = np.array([_cifar.targets[i] for i in idx])
    return LabelledSubset(idx, lbl)

def make_random_subset(size: int, num_classes: int = 10, seed: int = 42) -> LabelledSubset:

    rng = np.random.RandomState(seed)
    idx = rng.choice(len(_cifar), size=size, replace=False)
    lbl = rng.randint(0, num_classes, size=size)
    return LabelledSubset(idx, lbl)

def make_unique_subset(size, seed=42):
    rng = np.random.RandomState(seed)
    idx = rng.choice(len(_cifar), size=size, replace=False)
    lbl = np.arange(size)
    return LabelledSubset(idx, lbl)

def make_loader(subset: LabelledSubset) -> DataLoader:
    return DataLoader(
        subset,
        batch_size=min(CONFIG['batch_size'], len(subset)),
        shuffle=True,
        num_workers=CONFIG['num_workers'],
        pin_memory=True,
        drop_last=False, 
    )

@torch.no_grad()
def get_all_images(subset: LabelledSubset) -> torch.Tensor:

    loader = DataLoader(subset, batch_size=512, shuffle=False,
                        num_workers=CONFIG['num_workers'])
    return torch.cat([imgs for imgs, _ in loader], dim=0)
def make_unconditional_subset(size: int, seed: int = 42) -> LabelledSubset:

    rng = np.random.RandomState(seed)
    idx = rng.choice(len(_cifar), size=size, replace=False)
    lbl = np.zeros(size, dtype=np.int64)
    return LabelledSubset(idx, lbl)

# 6. Model

In [ ]:
class SinusoidalEmb(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 4), nn.SiLU(),
            nn.Linear(dim * 4, dim),
        )

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        half  = self.dim // 2
        freqs = torch.exp(-math.log(10000) *
                          torch.arange(half, device=t.device) / max(half - 1, 1))
        args  = t[:, None].float() * freqs[None]
        emb   = torch.cat([args.sin(), args.cos()], dim=-1)
        return self.mlp(emb)

class ResBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, t_dim: int):
        super().__init__()
        self.norm1  = nn.GroupNorm(min(8, out_ch), out_ch)
        self.norm2  = nn.GroupNorm(min(8, out_ch), out_ch)
        self.conv1  = nn.Conv2d(in_ch,  out_ch, 3, padding=1)
        self.conv2  = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.t_proj = nn.Linear(t_dim, out_ch)
        self.act    = nn.SiLU()
        self.skip   = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        h = self.act(self.norm1(self.conv1(x)))
        h = h + self.t_proj(t)[:, :, None, None]
        h = self.act(self.norm2(self.conv2(h)))
        return h + self.skip(x)

class DownBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, t_dim: int):
        super().__init__()
        self.res  = ResBlock(in_ch, out_ch, t_dim)
        self.down = nn.AvgPool2d(2)

    def forward(self, x, t):
        x = self.res(x, t)
        return self.down(x), x 

class UpBlock(nn.Module):
    def __init__(self, in_ch: int, skip_ch: int, out_ch: int, t_dim: int):
        super().__init__()
        self.up  = nn.ConvTranspose2d(in_ch, in_ch, kernel_size=2, stride=2)
        self.res = ResBlock(in_ch + skip_ch, out_ch, t_dim)

    def forward(self, x, skip, t):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        return self.res(x, t)

class UNet(nn.Module):
    def __init__(self, label_dim: int = 10, base: int = 128):
        super().__init__()
        t_dim = base

        self.time_emb  = SinusoidalEmb(t_dim)
        self.label_emb = nn.Embedding(label_dim, t_dim)

        self.down1 = DownBlock(3,        base,     t_dim)
        self.down2 = DownBlock(base,     base * 2, t_dim)
        self.bot   = ResBlock( base * 2, base * 4, t_dim)
        self.up1   = UpBlock(  base * 4, base * 2, base * 2, t_dim)
        self.up2   = UpBlock(  base * 2, base,     base,     t_dim)
        self.out   = nn.Sequential(nn.GroupNorm(8, base), nn.SiLU(),
                                   nn.Conv2d(base, 3, 1))

    def forward(self, x: torch.Tensor, t: torch.Tensor,
                labels: torch.Tensor) -> torch.Tensor:
        c = self.time_emb(t) + self.label_emb(labels)   
        x, s1 = self.down1(x, c)
        x, s2 = self.down2(x, c)
        x     = self.bot(x, c)
        x     = self.up1(x, s2, c)
        x     = self.up2(x, s1, c)
        return self.out(x)

class UNetUnconditional(nn.Module):

    def __init__(self, base: int = 128):
        super().__init__()
        t_dim = base

        self.time_emb = SinusoidalEmb(t_dim)

        self.down1 = DownBlock(3,        base,     t_dim)
        self.down2 = DownBlock(base,     base * 2, t_dim)
        self.bot   = ResBlock( base * 2, base * 4, t_dim)
        self.up1   = UpBlock(  base * 4, base * 2, base * 2, t_dim)
        self.up2   = UpBlock(  base * 2, base,     base,     t_dim)
        self.out   = nn.Sequential(nn.GroupNorm(8, base), nn.SiLU(),
                                   nn.Conv2d(base, 3, 1))

    def forward(self, x: torch.Tensor, t: torch.Tensor,
                labels=None) -> torch.Tensor:
        c = self.time_emb(t)
        x, s1 = self.down1(x, c)
        x, s2 = self.down2(x, c)
        x     = self.bot(x, c)
        x     = self.up1(x, s2, c)
        x     = self.up2(x, s1, c)
        return self.out(x)

# 7. EMA

In [ ]:
def ema_beta_for_dataset(dataset_size: int, epochs: int,
                          batch_size: int, halflife_steps: int = 500) -> float:
    steps_per_epoch = max(1, math.ceil(dataset_size / batch_size))
    total_steps     = steps_per_epoch * epochs
    beta = 0.5 ** (1.0 / max(halflife_steps, 1))
    beta = min(beta, 0.9999)
    print(f'  EMA beta={beta:.6f} | total steps={total_steps} | halflife={halflife_steps} steps')
    return beta

# 8. Training

In [ ]:
def train(
    subset: LabelledSubset,
    label_dim: int,
    ckpt_epochs: list = None,
    desc: str = '',
):
    ckpt_epochs = ckpt_epochs or []
    loader      = make_loader(subset)
    epochs      = CONFIG['epochs']

    model = UNet(label_dim=label_dim, base=CONFIG['base_channels']).to(device)
    ema   = copy.deepcopy(model).eval().requires_grad_(False)
    opt   = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])

    beta_ema    = ema_beta_for_dataset(len(subset), epochs, CONFIG['batch_size'])
    checkpoints = {}

    print(f'Training {desc} | |D|={len(subset)} | label_dim={label_dim} | '
          f'steps/epoch={len(loader)} | total steps={len(loader)*epochs}')

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0

        for imgs, lbls in loader:
            imgs = imgs.to(device)
            lbls = lbls.to(device)

            t            = torch.randint(0, T, (imgs.size(0),), device=device)
            x_noisy, eps = q_sample(imgs, t)

            pred = model(x_noisy, t, lbls)
            loss = nn.functional.mse_loss(pred, eps)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

            for p_ema, p in zip(ema.parameters(), model.parameters()):
                p_ema.data.lerp_(p.detach().data, 1.0 - beta_ema)

            running_loss += loss.item()

        if epoch % 100 == 0 or epoch == 1:
            avg = running_loss / len(loader)
            print(f'  [{desc}] epoch {epoch:4d}/{epochs}  loss={avg:.4f}')

        if epoch in ckpt_epochs:
            checkpoints[epoch] = copy.deepcopy(ema).cpu()
            print(f'  --> Checkpoint saved at epoch {epoch}')

    return ema.cpu(), checkpoints

def train_unconditional(
    subset: LabelledSubset,
    ckpt_epochs: list = None,
    desc: str = '',
):

    ckpt_epochs = ckpt_epochs or []
    loader      = make_loader(subset)
    epochs      = CONFIG['epochs']

    model = UNetUnconditional(base=CONFIG['base_channels']).to(device)
    ema   = copy.deepcopy(model).eval().requires_grad_(False)
    opt   = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])

    beta_ema    = ema_beta_for_dataset(len(subset), epochs, CONFIG['batch_size'])
    checkpoints = {}

    print(f'Training {desc} | |D|={len(subset)} | UNCONDITIONAL | '
          f'steps/epoch={len(loader)} | total steps={len(loader)*epochs}')

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0

        for imgs, _ in loader:
            imgs = imgs.to(device)

            t            = torch.randint(0, T, (imgs.size(0),), device=device)
            x_noisy, eps = q_sample(imgs, t)

            pred = model(x_noisy, t)
            loss = nn.functional.mse_loss(pred, eps)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

            for p_ema, p in zip(ema.parameters(), model.parameters()):
                p_ema.data.lerp_(p.detach().data, 1.0 - beta_ema)

            running_loss += loss.item()

        if epoch % 100 == 0 or epoch == 1:
            avg = running_loss / len(loader)
            print(f'  [{desc}] epoch {epoch:4d}/{epochs}  loss={avg:.4f}')

        if epoch in ckpt_epochs:
            checkpoints[epoch] = copy.deepcopy(ema).cpu()
            print(f'  --> Checkpoint saved at epoch {epoch}')

    return ema.cpu(), checkpoints

# 9. Sampling

In [ ]:
@torch.no_grad()
def sample_images(model: UNet, n: int, label_dim: int) -> torch.Tensor:
    model = model.to(device).eval()
    x      = torch.randn(n, 3, 32, 32, device=device)
    if label_dim <= n:
        labels = torch.arange(n, device=device) % label_dim
    else:
        labels = torch.randint(0, label_dim, (n,), device=device)

    for t_idx in tqdm(reversed(range(T)), desc='Sampling', total=T, leave=False):
        t_batch = torch.full((n,), t_idx, device=device, dtype=torch.long)
        eps_pred = model(x, t_batch, labels)

        a  = alpha[t_idx]
        ah = alpha_hat[t_idx]
        b  = beta[t_idx]

        x = (1.0 / a.sqrt()) * (x - (1.0 - a) / (1.0 - ah).sqrt() * eps_pred)
        if t_idx > 0:
            x = x + b.sqrt() * torch.randn_like(x)

    return x.clamp(-1.0, 1.0)

@torch.no_grad()
def sample_images_unconditional(model: UNetUnconditional, n: int) -> torch.Tensor:

    model = model.to(device).eval()
    x     = torch.randn(n, 3, 32, 32, device=device)

    for t_idx in tqdm(reversed(range(T)), desc='Sampling (uncond)', total=T, leave=False):
        t_batch  = torch.full((n,), t_idx, device=device, dtype=torch.long)
        eps_pred = model(x, t_batch)

        a  = alpha[t_idx]
        ah = alpha_hat[t_idx]
        b  = beta[t_idx]

        x = (1.0 / a.sqrt()) * (x - (1.0 - a) / (1.0 - ah).sqrt() * eps_pred)
        if t_idx > 0:
            x = x + b.sqrt() * torch.randn_like(x)

    return x.clamp(-1.0, 1.0)

# 10. Memorization Metric

In [ ]:
def memorization_ratio(
    model: UNet,
    subset: LabelledSubset,
    label_dim: int,
    threshold: float = 1/3,
):
    N_gen   = CONFIG['num_generated']
    gen     = sample_images(model, N_gen, label_dim)
    train   = get_all_images(subset).to(device)

    _, C, H, W = gen.shape
    norm        = math.sqrt(H * W * C)

    gen_f   = gen.view(N_gen, -1)
    train_f = train.view(train.size(0), -1)
    dist    = torch.cdist(gen_f, train_f) / norm

    k = min(2, train.size(0))
    if k < 2:
        print('Warning: need at least 2 training images.')
        return 0.0, 1.0

    top2, _ = torch.topk(dist, k=2, largest=False, dim=1)
    d1, d2  = top2[:, 0], top2[:, 1]

    ratio_per_sample = d1 / d2.clamp(min=1e-9)
    binary_ratio = (ratio_per_sample < threshold).float().mean().item()
    avg_ratio    = ratio_per_sample.mean().item()

    return binary_ratio, avg_ratio

def memorization_ratio_unconditional(
    model: UNetUnconditional,
    subset: LabelledSubset,
    threshold: float = 1/3,
):

    N_gen  = CONFIG['num_generated']
    gen    = sample_images_unconditional(model, N_gen)
    train  = get_all_images(subset).to(device)

    _, C, H, W = gen.shape
    norm        = math.sqrt(H * W * C)

    gen_f   = gen.view(N_gen, -1)
    train_f = train.view(train.size(0), -1)
    dist    = torch.cdist(gen_f, train_f) / norm

    k = min(2, train.size(0))
    if k < 2:
        print('Warning: need at least 2 training images.')
        return 0.0, 1.0

    top2, _ = torch.topk(dist, k=2, largest=False, dim=1)
    d1, d2  = top2[:, 0], top2[:, 1]

    ratio_per_sample = d1 / d2.clamp(min=1e-9)
    binary_ratio     = (ratio_per_sample < threshold).float().mean().item()
    avg_ratio        = ratio_per_sample.mean().item()

    return binary_ratio, avg_ratio

# Experiment A: Conditioning Type

In [ ]:
results_A = {}

subset_uncond = make_unconditional_subset(EXP_A_SIZE)
ema_uncond, _ = train_unconditional(subset_uncond, desc='A-uncond')
binary, avg = memorization_ratio_unconditional(ema_uncond, subset_uncond)
results_A['uncond'] = {'binary': binary, 'avg': avg}
print(f'  uncond  binary={binary:.4f}  avg_ratio={avg:.4f}')

for cond in ['normal', 'random', 'unique']:
    if cond == 'normal':
        subset    = make_normal_subset(EXP_A_SIZE)
        label_dim = CONFIG['num_classes']
    elif cond == 'random':
        subset    = make_random_subset(EXP_A_SIZE, num_classes=CONFIG['num_classes'])
        label_dim = CONFIG['num_classes']
    elif cond == 'unique':
        subset    = make_unique_subset(EXP_A_SIZE)
        label_dim = EXP_A_SIZE

    ema, _ = train(subset, label_dim=label_dim, desc=f'A-{cond}')
    binary, avg = memorization_ratio(ema, subset, label_dim)
    results_A[cond] = {'binary': binary, 'avg': avg}
    print(f'  {cond:8s}  binary={binary:.4f}  avg_ratio={avg:.4f}')

print('\n=== Experiment A complete ===')
for cond, r in results_A.items():
    print(f'  {cond:10s}  binary={r["binary"]:.4f}  avg_ratio={r["avg"]:.4f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

conds   = ['uncond', 'normal', 'random', 'unique']
labels  = ['Unconditional', 'Normal labels', 'Random labels', 'Unique labels']
colors  = ['mediumpurple', 'steelblue', 'coral', 'mediumseagreen']
binary_vals = [results_A[c]['binary'] * 100 for c in conds]
avg_vals    = [results_A[c]['avg']          for c in conds]

bars1 = ax1.bar(labels, binary_vals, color=colors, width=0.5)
ax1.bar_label(bars1, fmt='%.1f%%', padding=3)
ax1.set_ylabel('Memorisation ratio (%)')
ax1.set_title(f'Exp A — Binary memorisation\n(|D|={EXP_A_SIZE})')

bars2 = ax2.bar(labels, avg_vals, color=colors, width=0.5)
ax2.bar_label(bars2, fmt='%.3f', padding=3)
ax2.set_ylabel('Mean d1/d2 ratio (lower = more memorised)')
ax2.set_title(f'Exp A — Mean proximity ratio\n(|D|={EXP_A_SIZE})')
ax2.invert_yaxis()

fig.tight_layout()
fig.savefig('exp_A.png', dpi=150)
plt.show()

# Experiment B: Dataset Size

In [ ]:
results_B = {} 
models_B  = {}
subsets_B = {}

for size in EXP_B_SIZES:
    print(f'\n{"="*60}')
    print(f'Experiment B  |  dataset size = {size}  |  UNIQUE labels')
    print(f'{"="*60}')

    subset    = make_unique_subset(size)
    label_dim = size               

    ema, _ = train(subset, label_dim=label_dim, desc=f'B|D|={size}')

    binary, avg = memorization_ratio(ema, subset, label_dim)
    results_B[size] = {'binary': binary, 'avg': avg}
    models_B[size]  = ema
    subsets_B[size] = subset
    print(f'  --> Memorisation ratio (binary): {binary:.4f}')
    print(f'  --> Mean d1/d2 ratio:            {avg:.4f}  (lower = more memorised)')

print('\n=== Experiment B complete ===')
for size, r in results_B.items():
    print(f'  |D|={size:5d}  binary={r["binary"]:.4f}  avg_ratio={r["avg"]:.4f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

sizes  = list(results_B.keys())
binary = [results_B[s]['binary'] * 100 for s in sizes]
avg    = [results_B[s]['avg'] for s in sizes]

ax1.plot(sizes, binary, marker='o', linewidth=2, color='steelblue')
ax1.set_xlabel('Dataset size |D|')
ax1.set_ylabel('Memorisation ratio (%)')
ax1.set_title('Exp B — Binary memorisation\n(d1/d2 < 1/3, unique labels)')

ax2.plot(sizes, avg, marker='s', linewidth=2, color='coral')
ax2.set_xlabel('Dataset size |D|')
ax2.set_ylabel('Mean d1/d2 ratio (lower = more memorised)')
ax2.set_title('Exp B — Mean proximity ratio\n(softer metric, unique labels)')
ax2.invert_yaxis() 

fig.tight_layout()
fig.savefig('exp_B.png', dpi=150)
plt.show()

# Experiment C: Training Time

In [ ]:
print(f'\n{"="*60}')
print(f'Experiment C  |  dataset size = {EXP_C_SIZE}  |  UNIQUE labels')
print(f'{"="*60}')

subset_C    = make_unique_subset(EXP_C_SIZE, seed=123)
label_dim_C = EXP_C_SIZE

_, checkpoints_C = train(
    subset_C,
    label_dim=label_dim_C,
    ckpt_epochs=CONFIG['ckpt_epochs'],
    desc='C',
)

results_C = {}
for epoch in CONFIG['ckpt_epochs']:
    ckpt = checkpoints_C[epoch]
    binary, avg = memorization_ratio(ckpt, subset_C, label_dim_C)
    results_C[epoch] = {'binary': binary, 'avg': avg}
    print(f'  Epoch {epoch:5d}  binary={binary:.4f}  avg_ratio={avg:.4f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
epochs_c = list(results_C.keys())

ax1.plot(epochs_c, [results_C[e]['binary']*100 for e in epochs_c],
         marker='o', linewidth=2, color='steelblue')
ax1.set_xlabel('Training epoch')
ax1.set_ylabel('Memorisation ratio (%)')
ax1.set_title(f'Exp C — Binary memorisation over time\n(|D|={EXP_C_SIZE}, unique labels)')

ax2.plot(epochs_c, [results_C[e]['avg'] for e in epochs_c],
         marker='s', linewidth=2, color='coral')
ax2.set_xlabel('Training epoch')
ax2.set_ylabel('Mean d1/d2 ratio')
ax2.set_title(f'Exp C — Mean proximity ratio over time\n(|D|={EXP_C_SIZE}, unique labels)')
ax2.invert_yaxis()

fig.tight_layout()
fig.savefig('exp_C.png', dpi=150)
plt.show()

# Experiment D: Varying Number of Random Classes

In [ ]:
import numpy as np

_rng_e     = np.random.RandomState(42)
_exp_d_idx = _rng_e.choice(len(_cifar), size=EXP_D_SIZE, replace=False)

results_D = {}  

for C in EXP_D_C_VALS:
    print(f'\n{"="*60}')
    print(f'Experiment D  |  C = {C} random classes  |  size = {EXP_D_SIZE}')
    print(f'{"="*60}')

    rng_lbl = np.random.RandomState(42 + C) 
    if C == EXP_D_SIZE:
        lbl = np.arange(EXP_D_SIZE)
    else:
        lbl = rng_lbl.randint(0, C, size=EXP_D_SIZE)

    subset    = LabelledSubset(_exp_d_idx, lbl)
    label_dim = C

    ema, _ = train(subset, label_dim=label_dim, desc=f'D-C{C}')
    binary, avg = memorization_ratio(ema, subset, label_dim)
    results_D[C] = {'binary': binary, 'avg': avg}
    print(f'  --> binary={binary:.4f}  avg_ratio={avg:.4f}')

print('\n=== Experiment D complete ===')
for C, r in results_D.items():
    print(f'  C={C:5d}  binary={r["binary"]:.4f}  avg_ratio={r["avg"]:.4f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

c_vals      = list(results_D.keys())
binary_vals = [results_D[C]['binary'] * 100 for C in c_vals]
avg_vals    = [results_D[C]['avg']           for C in c_vals]
x_labels    = [str(C) for C in c_vals]

ax1.plot(x_labels, binary_vals, marker='o', linewidth=2, color='steelblue')
ax1.set_xlabel('Number of random classes (C)')
ax1.set_ylabel('Memorisation ratio (%)')
ax1.set_title(f'Exp D — Binary memorisation vs C\n(|D|={EXP_D_SIZE}, random labels)')
ax1.set_ylim(0, 105)

ax2.plot(x_labels, avg_vals, marker='s', linewidth=2, color='coral')
ax2.set_xlabel('Number of random classes (C)')
ax2.set_ylabel('Mean d1/d2 ratio (lower = more memorised)')
ax2.set_title(f'Exp D — Mean proximity ratio vs C\n(|D|={EXP_D_SIZE}, random labels)')
ax2.invert_yaxis()

fig.tight_layout()
fig.savefig('exp_D.png', dpi=150)
plt.show()

print('\n=== Experiment D Summary ===')
print(f'{"C":>8}  {"Binary":>8}  {"Avg ratio":>10}')
print('-' * 32)
for C, r in results_D.items():
    print(f'{C:>8}  {r["binary"]*100:>7.1f}%  {r["avg"]:>10.4f}')